In [ ]:
%load_ext autoreload
%autoreload 3 --print --log

# 从项目根目录或 examples 目录启动均可；统一以项目根目录运行。
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (cwd, *cwd.parents)
     if (p / "mtp_initializer").is_dir() and (p / "PROJECT.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("请从 scipykit 项目根目录或 examples 目录启动 notebook")
os.chdir(PROJECT_ROOT)
# 根目录用于本地脚本；父目录用于 import scipykit。同步对子进程生效。
python_paths = [str(PROJECT_ROOT), str(PROJECT_ROOT.parent)]
for path in reversed(python_paths):
    if path not in sys.path:
        sys.path.insert(0, path)
os.environ["PYTHONPATH"] = os.pathsep.join(
    dict.fromkeys(python_paths + [p for p in os.environ.get("PYTHONPATH", "").split(os.pathsep) if p])
)
os.environ["NOTEBOOK_NAME"] = "02_图像与自动标注"
print("项目路径:", PROJECT_ROOT)
print("当前解释器:", sys.executable)


# 图像、标注框和自动避让文字
`task.cv` 只组合绘图与图像 IO 接口。Camera/Frustum 旧实现已移除；相机几何使用独立的 graphmap 库。例子生成合成图像，不需要下载数据。

In [ ]:
from scipykit.task.cv import *
# 聚合导入同样自动应用历史 rcParams。
# 创建 RGB 图像；OpenCV 默认 BGR，读写时显式声明转换。
image = np.zeros((180, 320, 3), dtype=np.uint8)
image[..., 0] = np.linspace(20, 220, 320).astype(np.uint8)
image[..., 1] = 70
image[..., 2] = 110
path = Path("assets/02_图像与自动标注/中文图像.png")
cv2_imwrite(path, image, comp_ratio=3, is_from_RGB=True)
loaded = cv2_imread(path, is_to_RGB=True)
assert np.array_equal(loaded, image)
show_image(loaded, key="原图")

In [ ]:
fig, ax = subplots(figsize=(6.4, 3.6), layout="constrained")
ax.imshow(loaded)
boxes = [[25, 25, 125, 120], [170, 50, 290, 150]]
patches = mark_on_axes(ax, boxes, edgecolor=palette()[:2], facecolor=with_alpha("white", 0.12))
# exact 适合任意字体；mono_fast 适合等宽字体，速度更快。
labels = text_better(
    fig, ax, [75, 230], [70, 100], ["目标 A", "目标 B"],
    placement_mode="exact", color=["white", "yellow"],
    bbox={"boxstyle": "round,pad=0.25", "facecolor": "#222222", "alpha": 0.8},
    placement_kwargs={"radii_px": (15, 30, 50), "n_restarts": 2},
)
ax.set_axis_off()
disp(fig, "标注结果", embed=True)
plt.close(fig)

## OpenCV 标注与数据副本
`draw_box`、`draw_text`、`draw_pts` 不改变原数组，坐标采用 `xyxy`/`xy`。颜色跟随输入数组的通道顺序，默认按 OpenCV 的 BGR 习惯提供。OpenCV 内置字体只用于英文；中文使用 Matplotlib 文本。

In [ ]:
annotated = draw_box(loaded, boxes, color=(255, 200, 20))
annotated = draw_pts(annotated, [[75, 70], [230, 100]], color=(0, 255, 100), radius=5)
annotated = draw_text(annotated, ["A", "B"], boxes, color=(255, 255, 255), fontScale=1.5)
rgba = show_image(annotated, is_output=True, key="OpenCV标注")
print("导出的 RGBA:", rgba.shape)
assert np.array_equal(loaded, image)

## 自动标注扩展
`avoid_artists` 接受对象、类型或标签正则；`allow_artists` 可排除障碍物。已有 `find_annotate_position`、`find_annotate_positions`、`select_artists` 接口保留。批量标签会联合避让。不同字体或很密集的图优先用 `placement_mode="exact"`；更复杂的布局参数见函数帮助。